1. Project Initialization & Data Ingestion
Objective:
- Essential data manipulation aur visualization libraries (`pandas`, `numpy`, `matplotlib`, `seaborn`) ko load karna.
- Original raw dataset (`raw_data.csv`) ko memory mein load karna aur uski ek independent copy create karna taaki original raw data intact rahe.
- Data profiling ke zariye dataset ka preliminary **Data Quality Audit** run karna (null counts, data types, duplicate count, unique values check).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Raw dataset load karna
# Path '..' use kiya hai kyunki notebook 'notebooks/' folder ke andar hai aur data 'data/' folder mein
df_raw = pd.read_csv('../data/raw_data.csv')

# Backup copy create karna
df = df_raw.copy()

# 2. Initial Data Quality Audit Function
def generate_quality_report(data):
    report = pd.DataFrame({
        'Data Type': data.dtypes,
        'Null Count': data.isnull().sum(),
        'Null %': (data.isnull().sum() / len(data)) * 100,
        'Unique Values': data.nunique()
    })
    print("=" * 45)
    print(f"Total Rows      : {data.shape[0]}")
    print(f"Total Columns   : {data.shape[1]}")
    print(f"Duplicate Rows  : {data.duplicated().sum()}")
    print("=" * 45)
    return report

# Quality report display karna
generate_quality_report(df)

Total Rows      : 7500
Total Columns   : 16
Duplicate Rows  : 1237


,Data Type,Null Count,Null %,Unique Values
Location,str,0,0.0,267
Area,str,0,0.0,240
Baths,str,0,0.0,11
Beds,str,0,0.0,13
Dining Room,int64,0,0.0,2
Laundry Room,int64,0,0.0,2
Store Rooms,int64,0,0.0,13
Kitchens,int64,0,0.0,8
Drawing Room,int64,0,0.0,2
Gym,int64,0,0.0,2


Observations from Initial Quality Audit:
- **Missing Values:** Jin columns mein `Null Count > 0` hai, unke liye aage appropriate imputation strategy select karni hogi.
- **Data Types:** Check karein ki dates `datetime64`, categories `category`/`object`, aur identifiers `string` type mein hain ya nahi.
- **Duplicates:** Agar Duplicate Rows count zero se zyada hai, toh agle step mein unhe drop kiya jayega.

# 2. Duplicate Detection and Removal

### Strategy & Justification:
- Identical rows statistical metrics ko artificially skew kar deti hain (jaise mean aur counts).
- Ham complete duplicate rows ko identify karenge, exact count document karenge, aur unhe dataset se purge karenge.
- Original ordering retain karne ke liye index ko reset karenge.

In [2]:
# Initial row count note karna
initial_rows = len(df)
duplicate_count = df.duplicated().sum()

# Duplicates drop karna aur index reset karna
df = df.drop_duplicates().reset_index(drop=True)
remaining_rows = len(df)

print(f"Duplicates Detected & Removed: {duplicate_count}")
print(f"Row count transition: {initial_rows} -> {remaining_rows}")

Duplicates Detected & Removed: 1237
Row count transition: 7500 -> 6263


# 3. String & Categorical Standardisation

### Strategy & Justification:
- Messy datasets mein aksar leading/trailing whitespaces (`" Male "`) aur inconsistent casing (`"female"`, `"Female"`, `"F"`) hote hain.
- Pehle saare text (`object`) columns ko strip karke unke spaces clean karenge.
- Uske baad common categorical inconsistencies ko dictionary mapping ke zariye single standard format mein map karenge.

In [3]:
# 1. Sabhi text columns se unwanted spaces remove karna
text_columns = df.select_dtypes(include=['object']).columns

for col in text_columns:
    df[col] = df[col].astype(str).str.strip()

# 2. Example: Inconsistent gender values standardise karna
# (Agar aapke dataset mein 'gender' ya 'sex' column hai)
gender_col = next((c for c in df.columns if c.lower() in ['gender', 'sex']), None)

if gender_col:
    print(f"Unique values before standardisation in '{gender_col}':", df[gender_col].unique())
    
    mapping = {
        'male': 'Male', 'm': 'Male', 'M': 'Male', 'MALE': 'Male',
        'female': 'Female', 'f': 'Female', 'F': 'Female', 'FEMALE': 'Female'
    }
    df[gender_col] = df[gender_col].map(mapping).fillna(df[gender_col])
    print(f"Unique values after standardisation in '{gender_col}':", df[gender_col].unique())
else:
    print("No gender column found, string trimming applied across all text columns.")

No gender column found, string trimming applied across all text columns.


C:\Users\Janhavi\AppData\Local\Temp\ipykernel_17308\1945655081.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = df.select_dtypes(include=['object']).columns


# 4. Missing Data Strategy & Execution

### Imputation Rules & Justification:
1. **Numeric Columns (Median Imputation):** Outliers ya skewed distribution hone par `mean` biased ho jata hai, isliye robust central tendency ke roop mein `median` use kiya gaya hai.
2. **Categorical Columns (Mode Imputation):** Discrete textual data ke liye most frequent valid value (`mode`) fill ki gayi hai.
3. **Primary Identifiers / Target (Drop Row):** Agar unique ID ya target label missing ho toh synthetic value dalna data leakage ya bad training ka karan banta hai.

In [4]:
# Check missing values before handling
missing_before = df.isnull().sum()
print("Missing counts before handling:")
print(missing_before[missing_before > 0])

# 1. Numeric columns -> Median
num_cols = df.select_dtypes(include=[np.number]).columns
for col in num_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"Imputed column '{col}' with median: {median_val}")

# 2. Categorical columns -> Mode
cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0]
        df[col] = df[col].fillna(mode_val)
        print(f"Imputed column '{col}' with mode: {mode_val}")

print("\nRemaining missing values:", df.isnull().sum().sum())

Missing counts before handling:
Series([], dtype: int64)

Remaining missing values: 0


C:\Users\Janhavi\AppData\Local\Temp\ipykernel_17308\1611668257.py:15: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object']).columns


# 5. Domain-Specific Feature Transformation

### Strategy & Justification:
- **Price:** Values `'PKR3.5 Crore'` aur `'PKR85 Lakh'` string format mein hain. Inhe numeric float (PKR amount) mein convert kiya gaya:
  - 1 Crore = 10,000,000
  - 1 Lakh = 100,000
- **Area:** Mixed units `'Marla'` aur `'Kanal'` ko single standard unit (`Marla`) mein normalise kiya gaya (1 Kanal = 20 Marla).
- **Baths & Beds:** Missing values placeholder `'-'` ke roop mein hain. Inhe NaN banakar respective median se impute kiya gaya aur integer cast kiya gaya.

In [5]:
# 1. Price conversion function
def convert_price(val):
    if pd.isnull(val):
        return np.nan
    val = str(val).replace('PKR', '').strip()
    if 'Crore' in val:
        return float(val.replace('Crore', '').strip()) * 1e7
    elif 'Lakh' in val:
        return float(val.replace('Lakh', '').strip()) * 1e5
    return pd.to_numeric(val, errors='coerce')

df['Price_PKR'] = df['Price'].apply(convert_price)

# 2. Area conversion to standard 'Marla'
def convert_area_to_marla(val):
    if pd.isnull(val):
        return np.nan
    val = str(val).strip()
    if 'Kanal' in val:
        return float(val.replace('Kanal', '').strip()) * 20.0
    elif 'Marla' in val:
        return float(val.replace('Marla', '').strip())
    return pd.to_numeric(val, errors='coerce')

df['Area_Marla'] = df['Area'].apply(convert_area_to_marla)

# 3. Clean Baths & Beds ('-' to NaN -> Median Imputation)
for col in ['Baths', 'Beds']:
    df[col] = pd.to_numeric(df[col].replace('-', np.nan), errors='coerce')
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val).astype(int)

# Old raw text columns drop karna
df = df.drop(columns=['Price', 'Area'])

print("Feature transformation complete. New numeric columns created: 'Price_PKR', 'Area_Marla'")
print(df[['Baths', 'Beds', 'Area_Marla', 'Price_PKR']].head())

Feature transformation complete. New numeric columns created: 'Price_PKR', 'Area_Marla'
   Baths  Beds  Area_Marla    Price_PKR
0      6     5        10.0   35000000.0
1      4     3         4.0   13200000.0
2      4     3         5.0   17400000.0
3      6     5        20.0  103000000.0
4      6     5        20.0   82500000.0


# 6. Outlier Handling using Interquartile Range (IQR)

### Strategy:
- `Price_PKR`, `Area_Marla`, `Store Rooms`, aur `Kitchens` jaise numeric features mein extreme values (e.g., 32 Store Rooms ya 20 Kitchens) paye gaye hain jo real-world residential data ke distribution ko heavily skew kar dete hain.
- Ham **IQR (Interquartile Range) Capping (Winsorizing)** approach use karenge:
  - $IQR = Q3 - Q1$
  - $\text{Lower Bound} = Q1 - 1.5 \times IQR$
  - $\text{Upper Bound} = Q3 + 1.5 \times IQR$
- Data loss avoid karne ke liye rows delete karne ke bajaye upper aur lower bounds par values ko clip/cap kiya gaya hai.

In [6]:
# Target columns for outlier inspection
outlier_cols = ['Price_PKR', 'Area_Marla', 'Store Rooms', 'Kitchens', 'Baths', 'Beds']

def cap_outliers_iqr(dataframe, column):
    q1 = dataframe[column].quantile(0.25)
    q3 = dataframe[column].quantile(0.75)
    iqr = q3 - q1
    lower_bound = max(0, q1 - 1.5 * iqr)  # Real estate counts/price negative nahi ho sakte
    upper_bound = q3 + 1.5 * iqr
    
    outliers_count = ((dataframe[column] < lower_bound) | (dataframe[column] > upper_bound)).sum()
    print(f"{column:12s} | Outliers: {outliers_count:4d} | Bounds: [{lower_bound:.1f}, {upper_bound:.1f}]")
    
    dataframe[column] = dataframe[column].clip(lower=lower_bound, upper=upper_bound)
    return dataframe

print("Detecting and capping outliers:")
print("-" * 55)
for col in outlier_cols:
    df = cap_outliers_iqr(df, col)

Detecting and capping outliers:
-------------------------------------------------------
Price_PKR    | Outliers:  455 | Bounds: [0.0, 200000000.0]
Area_Marla   | Outliers:   94 | Bounds: [0.0, 42.5]
Store Rooms  | Outliers:  120 | Bounds: [0.0, 2.5]
Kitchens     | Outliers:    3 | Bounds: [0.0, 5.0]
Baths        | Outliers:    8 | Bounds: [1.0, 9.0]
Beds         | Outliers:   70 | Bounds: [1.0, 9.0]


# 7. Quality Audit & Clean Dataset Export

### Summary:
- Data quality aur integrity ka initial raw dataset se processed clean dataset tak direct comparison.
- Cleaned dataset ko analysis ke liye `data/cleaned_data.csv` par export karna.

In [7]:
summary_table = pd.DataFrame({
    'Metric': [
        'Total Rows',
        'Duplicate Rows',
        'Hidden Missing Values ("-")',
        'Price Column Status',
        'Area Column Status'
    ],
    'Before Cleaning': [
        len(df_raw),
        df_raw.duplicated().sum(),
        (df_raw[['Baths', 'Beds']] == '-').sum().sum(),
        'Unusable Text (e.g. PKR3.5 Crore)',
        'Mixed Units (Kanal & Marla)'
    ],
    'After Cleaning': [
        len(df),
        df.duplicated().sum(),
        0,
        'Clean Numeric (Float PKR)',
        'Standardised (Marla)'
    ]
})

print("=== FINAL DATA QUALITY TRANSFORMATION AUDIT ===")
print(summary_table.to_string(index=False))

# Export clean file
df.to_csv('../data/cleaned_data.csv', index=False)
print("\n[SUCCESS] Cleaned dataset exported to '../data/cleaned_data.csv'")

=== FINAL DATA QUALITY TRANSFORMATION AUDIT ===
                     Metric                   Before Cleaning            After Cleaning
                 Total Rows                              7500                      6263
             Duplicate Rows                              1237                       130
Hidden Missing Values ("-")                               837                         0
        Price Column Status Unusable Text (e.g. PKR3.5 Crore) Clean Numeric (Float PKR)
         Area Column Status       Mixed Units (Kanal & Marla)      Standardised (Marla)

[SUCCESS] Cleaned dataset exported to '../data/cleaned_data.csv'
